# 🗜️ Notebook 03: Multistage Pauli Correlation Encoding (`PCE`) — Sub-Linear Qubit Compression
**PravahaQ (`v0.1.0`) — Encoding $N \sim \mathcal{O}(n^k)$ Industrial Variables onto $n \ll N$ Qubits**

---

## 🟢 Layer 1: Plain-English Intuition (The Piano Chord Analogy)

What if your data center has **84 binary scheduling decisions**, but your high-fidelity quantum processor subgraph only has **7 physical qubits**?
- In a standard 1-to-1 QUBO mapping, 7 qubits can only represent **7 decisions** (1 note per piano key).
- In **Multistage Pauli Correlation Encoding (`PCE`)**, we assign each binary decision to a unique **2-qubit or 3-qubit Pauli chord** ($\hat{\Pi}_m \in \{I, X, Y, Z\}^{\otimes n}$).
- With just $n = 7$ qubits, there are $3\binom{7}{2} = 63$ two-body chords and $3\binom{7}{3} = 105$ three-body chords—allowing **7 physical qubits to encode over 168 binary variables simultaneously!**

---

## 🟣 Layer 3: Multistage Hyperbolic Tangent Curriculum Loss

Each binary spin $z_m \in \{-1, +1\}$ ($m = 1, \dots, N$) is relaxed to a smooth expectation correlator:
$$\tilde{z}_m(\boldsymbol{\theta}) = \tanh\!\left(\alpha_s \langle \psi(\boldsymbol{\theta}) | \hat{\Pi}_m | \psi(\boldsymbol{\theta}) \rangle\right)$$
optimized across curriculum stages $s \in \{1, 2, 3\}$ with increasing slope $\alpha_s$ and sign-saturation regularizer $\beta_s$:
$$\mathcal{L}_{\text{PCE}}(\boldsymbol{\theta}; \alpha_s, \beta_s) = E_{\text{Ising}}(\tilde{\mathbf{z}}(\boldsymbol{\theta})) + \beta_s \sum_{m=1}^{N} \left(1 - \tilde{z}_m(\boldsymbol{\theta})^2\right)$$

In [1]:
import sys
from math import comb
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

print("=== Pauli Correlator Compression Capacity Table (k <= 3) ===")
print("Physical Qubits (n) | 2-Body Chords | 3-Body Chords | Total Homogeneous Capacity (N_max)")
print("-" * 87)
for n_q in range(4, 9):
    c1 = 3 * n_q
    c2 = 3 * comb(n_q, 2)
    c3 = 3 * comb(n_q, 3)
    print(f"        {n_q} Q         |     {c2:3d}       |     {c3:3d}       |              {c1 + c2 + c3:3d} vars")

=== Pauli Correlator Compression Capacity Table (k <= 3) ===
Physical Qubits (n) | 2-Body Chords | 3-Body Chords | Total Homogeneous Capacity (N_max)
---------------------------------------------------------------------------------------
        4 Q         |      18       |      12       |               42 vars
        5 Q         |      30       |      30       |               75 vars
        6 Q         |      45       |      60       |              123 vars
        7 Q         |      63       |     105       |              189 vars
        8 Q         |      84       |     168       |              276 vars


In [2]:
from pravaha_q.domain import generate_coupled_problem
from pravaha_q.math_engine import VaisheshikaHUBOCompiler, hubo_to_ising, FeasibilityRepairEngine
from pravaha_q.encodings import MultistagePCEEncoder

# 4 Edge Nodes, 8 LLM Batches, 1 Power State => 36 Binary Decision Variables
large_prob = generate_coupled_problem(num_nodes=4, num_tasks=8, num_power_states=1, seed=108)
large_hubo = VaisheshikaHUBOCompiler().compile(large_prob)
large_ising = hubo_to_ising(large_hubo)

pce_solver = MultistagePCEEncoder(num_qubits=5, ansatz_layers=2, seed=108)
pce_res = pce_solver.solve(large_ising)

repair = FeasibilityRepairEngine(large_prob, large_hubo)
schedule = repair.repair_and_decode(pce_res.best_bitstring)

print("=== Compressing 36-Variable Industrial Cluster onto 5 Physical Qubits ===")
print(f"Total Binary Variables (N) : {pce_res.num_variables}")
print(f"Physical Qubits Used (n)   : {pce_res.num_qubits_used} (Compression Ratio: {pce_res.compression_ratio}x)")
print(f"Sample Pauli Correlators   : {pce_res.correlator_strings[:6]}")
print(f"PCE Decoded Energy         : {schedule.repaired_energy:.4f}")
print(f"Post-Repair SLA Feasible?  : {schedule.is_feasible} (Violations Repaired: {schedule.constraint_violations_fixed})")
print(f"Task -> Node Assignment    : {schedule.task_to_node}")

=== Compressing 36-Variable Industrial Cluster onto 5 Physical Qubits ===
Total Binary Variables (N) : 36
Physical Qubits Used (n)   : 5 (Compression Ratio: 7.2x)
Sample Pauli Correlators   : ['ZZIII', 'ZIZII', 'ZIIZI', 'ZIIIZ', 'IZZII', 'IZIZI']
PCE Decoded Energy         : -54.8210
Post-Repair SLA Feasible?  : True (Violations Repaired: 1)
Task -> Node Assignment    : {0: 1, 1: 0, 2: 2, 3: 3, 4: 1, 5: 0, 6: 2, 7: 3}
